# Prediction Interval Coverage Analysis for Smoothed Random Forests

This notebook analyzes the 95% prediction interval coverage for:
- Standard Random Forests (RF10, RF20, RF50, RF100)
- Gaussian Process (GP)
- Smoothed Random Forests (various strategies)

Well-calibrated models should achieve coverage $\approx$ 0.95

# Setup and Configuration




In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import os
import sys
from sklearn.metrics import mean_squared_error
import json
from scipy.stats import norm
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

In [6]:
DATASET_MAX_N_OBS = {
    'autompg': 400,
    'breastcancer': 200,
    'fertility': 100,  # Small dataset
    'forest': 500,
    'housing': 500,
    'pendulum': 500,
    'qsar_aquatic_toxicity': 500,
    'servo': 100,      # Small dataset
    'stock': 500,
    'yacht_hydrodynamics': 300 , # Medium dataset
    'ENB2012_data_energy_heating': 768,
    'ENB2012_data_energy_cooling': 768,
    'real_estate': 414,
    'winequality-red': 1599,
    'winequality-white': 4898,
    'airfoil_self_noise': 1503,
    'qsar_fish_toxicity': 908,
    'Combined_Cycle_Power_Plant': 9568,
}

def get_valid_n_obs_list(data_name, requested_n_obs_list):
    """
    Filter n_obs_list based on dataset's maximum available samples
    
    Args:
        data_name: Name of the dataset
        requested_n_obs_list: List of requested sample sizes
        
    Returns:
        Filtered list of valid sample sizes
    """
    max_n_obs = DATASET_MAX_N_OBS.get(data_name, 500)
    valid_list = [n for n in requested_n_obs_list if n <= max_n_obs]
    
    return valid_list

# List of datasets (same as in experiment)
data_names = [
    'fertility', 
    'forest',
    'qsar_aquatic_toxicity', 
    'stock', 
    'yacht_hydrodynamics',
    'real_estate',
    'winequality-red',
    'winequality-white',
    'qsar_fish_toxicity',
    'Combined_Cycle_Power_Plant'
]

def get_CI(y_pred, std, alpha=0.05):
    z = norm.ppf(1-alpha/2)
    CI = z*std
    return y_pred - CI, y_pred + CI

def get_CI_coverage(y_pred,y_true,y_std,alpha = 0.05):
    ci_low, ci_high = get_CI(y_pred,y_std,alpha)
    ci_cover_no = np.logical_and(ci_low<y_true,ci_high>y_true).astype(int).sum()
    return ci_cover_no/y_true.shape[0]

def log_loss(y_true, y_pred, y_std):
    # Ensure all inputs are numpy arrays for vectorized operations
    y_true = np.array(y_true)
    y_pred = np.array(y_pred, dtype=np.float64)
    y_std = np.array(y_std, dtype=np.float64)

    # Calculate the log PDF of the true values under the normal distributions defined by preds and stds
    logpdf_values = norm.logpdf(y_true, loc=y_pred, scale=y_std)

    # Filter out NaN values if any
    valid_logpdf_values = logpdf_values[~np.isnan(logpdf_values)]

    # Calculate and return the mean log loss
    return -np.mean(valid_logpdf_values)

In [7]:
def get_pred_vars(parent_path,data_name,n_obs,r):
    est_pd_pred_file = f'{parent_path}/{data_name}/EST_PD_predictions_noCV/{data_name}_n{n_obs}_r{r}.csv'
    other_mode_pred_file = f'{parent_path}/{data_name}/other_mode_predictions_noCV/{data_name}_n{n_obs}_r{r}.csv'
    baseline_pred_file = f'{parent_path}/{data_name}/predictions/{data_name}_n{n_obs}_r{r}.csv'

    est_pd_pred_df = pd.read_csv(est_pd_pred_file)
    other_mode_pred_df = pd.read_csv(other_mode_pred_file)
    baseline_pred_df = pd.read_csv(baseline_pred_file)

    return baseline_pred_df, est_pd_pred_df, other_mode_pred_df

In [8]:
def metrics(y_true,y_pred,y_std,metric='MSE'):
    if metric == 'MSE':
        return mean_squared_error(y_pred=y_pred,y_true=y_true)
    elif metric =='ci_coverage':
        return get_CI_coverage(y_pred=y_pred,y_true=y_true,y_std=y_std)
    elif metric == 'logloss':
        return log_loss(y_true=y_true,y_pred=y_pred,y_std=y_std)
    else:
        raise ValueError(f"Metric {metric} not supported")

In [9]:
def add_data_info(data_names):
    data_info = pd.DataFrame()
    p_list = []
    n_list = []
    for data_name in data_names:
        data = pd.read_csv(f'../../data/{data_name}.csv')
        p_list.append(data.shape[1]-1)
        n_list.append(data.shape[0])
    data_info['data'] = data_names
    data_info['P'] = p_list
    data_info['N'] = n_list
    return data_info
    

In [10]:
data_info_df = add_data_info(data_names)

In [11]:
dataset_info = {
    data:{'n':data_info_df[data_info_df['data'] == data].N.item(),'p':data_info_df[data_info_df['data'] == data].P.item()} for data in data_info_df['data']
}

In [12]:
def calculate_metrics(data_name, dataset_info,n_obs_list_full = [50,100,200,300,400,500], 
                            results_dir='../results', metrics_dir='./metrics',metric='MSE'):
    """
    Calculate MSE for all models across different sample sizes and replications
    
    Args:
        data_name: Dataset name (e.g., 'autompg')
        n_obs_list: List of sample sizes to analyze
        save: Whether to save results to CSV
        results_dir: Directory containing experiment results
        metrics_dir: Directory to save metrics (if None, won't save even if save=True)
    
    Returns:
        DataFrame with columns: data, n_obs, r, base_rf, rf10, rf100, srf_normal_*, srf_hypsec_*
    """
    n_obs_list = get_valid_n_obs_list(data_name, n_obs_list_full)
    # List to store results
    results_list = []
    
    # Define all model names (use original names for reading)
    baseline_model_names = ['rf10', 'rf20', 'rf50', 'rf_100','gp']
    est_pd_model_names = ['srf_normal_EST_PD','srf_hypsec_EST_PD']
    other_mode_model_names = ['srf_normal_global','srf_hypsec_global','srf_normal_per_dim',
                        'srf_hypsec_per_dim','srf_normal_per_tree','srf_hypsec_per_tree']
    
    # Iterate through all sample sizes and replications
    for n_obs in n_obs_list:
        for r in range(100):  # 100 replications
            
            baseline_pred_file = f'{results_dir}/{data_name}/predictions/{data_name}_n{n_obs}_r{r}.csv'
            est_pd_pred_file = f'{results_dir}/{data_name}/EST_PD_predictions_noCV/{data_name}_n{n_obs}_r{r}.csv'
            other_mode_pred_file = f'{results_dir}/{data_name}/other_mode_predictions_noCV/{data_name}_n{n_obs}_r{r}.csv'
            
            # Check if file exists
            if not os.path.exists(baseline_pred_file):
                continue
            
            try:
                baseline_pred_df = pd.read_csv(baseline_pred_file)
                est_pd_pred_df = pd.read_csv(est_pd_pred_file)
                other_mode_pred_df = pd.read_csv(other_mode_pred_file)
                # Load predictions
                y_test = baseline_pred_df['y_test'].values
                
                # Create result dictionary for this replication
                result = {
                    'data': data_name,
                    'p': dataset_info[data_name]['p'],
                    'n_obs': n_obs,
                    'r': r
                }
                
                # Calculate MSE for each model
                for model in baseline_model_names:
                    base_pred_col = f'{model}_pred'
                    base_std_col = f'{model}_std'
                    
                    if base_pred_col in baseline_pred_df.columns:
                        base_pred = baseline_pred_df[base_pred_col].values
                        base_std = baseline_pred_df[base_std_col].values
                        
                        # Check for NaN values
                        if np.any(np.isnan(base_pred)):
                            result[model] = np.nan
                        else:
                            result[model] = metrics(y_true=y_test,y_pred=base_pred,y_std=base_std,metric=metric)
                    else:
                        result[model] = np.nan
                for est_pd_model in est_pd_model_names:
                    est_pd_pred_col = f'{est_pd_model}_pred'
                    est_pd_std_col = f'{est_pd_model}_total_std'
                    
                    if est_pd_pred_col in est_pd_pred_df.columns:
                        est_pd_pred = est_pd_pred_df[est_pd_pred_col].values
                        est_pd_std = est_pd_pred_df[est_pd_std_col].values
                        
                        # Check for NaN values
                        if np.any(np.isnan(est_pd_pred)):
                            result[est_pd_model] = np.nan
                        else:
                            result[est_pd_model] = metrics(y_true=y_test,
                                                        y_pred=est_pd_pred,
                                                        y_std=est_pd_std,
                                                        metric=metric)
                    else:
                        result[est_pd_model] = np.nan
                
                for other_mode_model in other_mode_model_names:
                    other_mode_pred_col = f'{other_mode_model}_pred'
                    other_mode_std_col = f'{other_mode_model}_total_std'
                    
                    if other_mode_pred_col in other_mode_pred_df.columns:
                        other_mode_pred = other_mode_pred_df[other_mode_pred_col].values
                        other_mode_std = other_mode_pred_df[other_mode_std_col].values
                        
                        # Check for NaN values
                        if np.any(np.isnan(other_mode_pred)):
                            result[other_mode_model] = np.nan
                        else:
                            result[other_mode_model] = metrics(y_true=y_test,
                                                        y_pred=other_mode_pred,
                                                        y_std=other_mode_std,
                                                        metric=metric)
                results_list.append(result)
                
            except Exception as e:
                print(f"Error processing {data_name} n={n_obs} r={r}: {e}")
                continue
    
    # Create DataFrame
    results_df = pd.DataFrame(results_list)
    # Rename rf_full to rf100
    model_names_display = [m if m != 'rf_100' else 'rf100' for m in baseline_model_names]
    # combine srf_normal_EST_PD and srf_hypsec_EST_PD into baseline_model_names
    results_df.rename(columns={'rf_100': 'rf100'}, inplace=True)
    model_names_display.append('srf_normal_EST_PD')
    model_names_display.append('srf_hypsec_EST_PD')
    model_names_display.append('srf_normal_global')
    model_names_display.append('srf_hypsec_global')
    model_names_display.append('srf_normal_per_dim')
    model_names_display.append('srf_hypsec_per_dim')
    model_names_display.append('srf_normal_per_tree')
    model_names_display.append('srf_hypsec_per_tree')
    # Reorder columns
    cols_order = ['data','p', 'n_obs', 'r'] + model_names_display
    results_df = results_df[cols_order]
    results_df.rename(columns={
        'data':'Data',
        'rf10':'RF(10)',
        'rf20':'RF(20)',
        'rf50':'RF(50)',
        'rf100': 'RF(100)',
        'gp':'GP',
        'srf_normal_EST_PD':'EST-PD(Norm)',
        'srf_hypsec_EST_PD':'EST-PD(Hypsec)',
        'srf_normal_global':'STE(Norm)',
        'srf_hypsec_global':'STE(Hypsec)',
        'srf_normal_per_dim':'STE-PD(Norm)',
        'srf_hypsec_per_dim':'STE-PD(Hypsec)',
        'srf_normal_per_tree':'EST(Norm)',
        'srf_hypsec_per_tree':'EST(Hypsec)'
        }, inplace=True)
    
    # Save if requested and metrics_dir is not None
    if metrics_dir is not None:
        save_path = f'{metrics_dir}/{data_name}/{data_name}_{metric}.csv'
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        results_df.to_csv(save_path, index=False)
        print(f"Saved metrics to: {save_path}")
    
    else:
        return results_df

In [13]:
for data in data_names:
    calculate_metrics(data_name=data,dataset_info=dataset_info,metrics_dir= './metrics',metric='ci_coverage')

Saved metrics to: ./metrics/fertility/fertility_ci_coverage.csv
Saved metrics to: ./metrics/forest/forest_ci_coverage.csv
Saved metrics to: ./metrics/qsar_aquatic_toxicity/qsar_aquatic_toxicity_ci_coverage.csv
Saved metrics to: ./metrics/stock/stock_ci_coverage.csv
Saved metrics to: ./metrics/yacht_hydrodynamics/yacht_hydrodynamics_ci_coverage.csv
Saved metrics to: ./metrics/real_estate/real_estate_ci_coverage.csv
Saved metrics to: ./metrics/winequality-red/winequality-red_ci_coverage.csv
Saved metrics to: ./metrics/winequality-white/winequality-white_ci_coverage.csv
Saved metrics to: ./metrics/qsar_fish_toxicity/qsar_fish_toxicity_ci_coverage.csv
Saved metrics to: ./metrics/Combined_Cycle_Power_Plant/Combined_Cycle_Power_Plant_ci_coverage.csv


In [14]:
for data in data_names:
    calculate_metrics(data_name=data,dataset_info=dataset_info,metrics_dir= './metrics',metric='logloss')

Saved metrics to: ./metrics/fertility/fertility_logloss.csv
Saved metrics to: ./metrics/forest/forest_logloss.csv
Saved metrics to: ./metrics/qsar_aquatic_toxicity/qsar_aquatic_toxicity_logloss.csv
Saved metrics to: ./metrics/stock/stock_logloss.csv
Saved metrics to: ./metrics/yacht_hydrodynamics/yacht_hydrodynamics_logloss.csv
Saved metrics to: ./metrics/real_estate/real_estate_logloss.csv
Saved metrics to: ./metrics/winequality-red/winequality-red_logloss.csv
Saved metrics to: ./metrics/winequality-white/winequality-white_logloss.csv
Saved metrics to: ./metrics/qsar_fish_toxicity/qsar_fish_toxicity_logloss.csv
Saved metrics to: ./metrics/Combined_Cycle_Power_Plant/Combined_Cycle_Power_Plant_logloss.csv


In [16]:
data_name_mapping = {
    'Combined_Cycle_Power_Plant': 'CCPP',
    'qsar_fish_toxicity': 'Qsar Fish Toxicity',
    'real_estate': 'Real Estate',
    'yacht_hydrodynamics': 'Yacht Hydrodynamics',
    'qsar_aquatic_toxicity': 'Qsar Aquatic Toxicity',
    'fertility': 'Fertility',
    'stock': 'Stock',
    'winequality-red': 'Winequality (Red)',
    'winequality-white': 'Winequality (White)',
    'forest': 'Forest'
    }

In [17]:
def combine_all_data_metrics(metric,data_name_mapping,data_list,metrics_path,save_path=None):
    metric_df = pd.DataFrame()
    for data_name in data_list:
        metric_df = pd.concat([metric_df,pd.read_csv(f'{metrics_path}/{data_name}/{data_name}_{metric}.csv')])


    metric_df['Data'] = metric_df['Data'].replace(data_name_mapping)
  
    if save_path is not None:
        metric_df.to_csv(f'{save_path}/{metric}_combined_all_data.csv',index=False)
    else:
        return metric_df

In [18]:
combine_all_data_metrics(metric='ci_coverage',data_list=data_names,data_name_mapping=data_name_mapping,
                                        metrics_path= './metrics',
                                        save_path= './metrics')

In [19]:
combine_all_data_metrics(metric='logloss',data_list=data_names,data_name_mapping = data_name_mapping,
                                        metrics_path= './metrics',
                                        save_path= './metrics')